# Setup & Dependencies

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames[0:5]:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data/label/20240529_EO4_RES2_fl_pid_074_label.tif
/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data/label/20240529_EO4_RES2_fl_pid_070_label.tif
/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data/label/20240529_EO4_RES2_fl_pid_035_label.tif
/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data/label/20240529_EO4_RES2_fl_pid_042_label.tif
/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data/label/20240529_EO4_RES2_fl_pid_012_label.tif
/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data/split/pred.txt
/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data/split/val.txt
/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data/split/test.txt
/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data/split/train.txt
/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data/split/.ipynb_checkpoints/train-checkpoint.txt
/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data/split/.ip

In [2]:
!pip install -q numpy pandas lightning segmentation-models-pytorch rasterio kagglehub torchvision timm scipy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 12.6 MB/s eta 0:00:00


# Imports & Configuration

In [3]:
import os
import math
from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import tv_tensors
from torchvision.transforms import v2
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor
import timm
import kagglehub
from scipy.ndimage import distance_transform_edt

# ─── Configuration ────────────────────────────────────────────────────────────
DATA_ROOT        = '/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data'
OUT_DIR          = '/kaggle/working/predictions'
MODEL_SAVE_DIR   = '/kaggle/working/model_export'
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

NUM_CLASSES = 3          # 0=Background, 1=Flood, 2=Water Body
IN_CHANNELS = 8          # STRIPPED DOWN: SAR_HH, SAR_HV, NDWI, SAR_ratio
PATCH_SIZE  = 512
EPOCHS      = 50
BATCH_SIZE  = 8          # We can bump this up since ResNet18 is much smaller!
LR          = 3e-4

# ─── Band layout in the raw 7-channel TIF ─────────────────────────────────────
BAND_SAR_HH = 0
BAND_SAR_HV = 1
BAND_GREEN  = 2
BAND_RED    = 3
BAND_NIR    = 4
BAND_SWIR   = 5

# Normalization constants for the raw bands
MEANS_6 = np.array([841.1257, 371.6175, 1734.1789, 1588.3142, 1742.8452, 1218.5551], dtype=np.float32)
STDS_6  = np.array([473.7090,  170.3611,  623.0474,  612.8465,  564.5835,  528.0894], dtype=np.float32)

# Define 8-channel constants by appending placeholders for the derived indices
MEANS_8 = np.concatenate([MEANS_6, [0.0, 0.0]]).astype(np.float32)
STDS_8  = np.concatenate([STDS_6,  [1.0, 1.0]]).astype(np.float32)

# # NEW: We only want [SAR_HH, SAR_HV, NDWI, SAR_Ratio]
# MEANS_4 = np.array([MEANS_6[0], MEANS_6[1], 0.0, 0.0], dtype=np.float32)
# STDS_4  = np.array([STDS_6[0],  STDS_6[1],  1.0, 1.0], dtype=np.float32)

print("Configuration loaded. (Domain-Shift Proof Mode)")
print(f"  Classes : {NUM_CLASSES}  |  Channels : {IN_CHANNELS} (SAR & Indices Only)")
print(f"  Epochs  : {EPOCHS}       |  Batch    : {BATCH_SIZE}")

Configuration loaded. (Domain-Shift Proof Mode)
  Classes : 3  |  Channels : 8 (SAR & Indices Only)
  Epochs  : 50       |  Batch    : 8


# Data Loading

In [4]:
train_ids, val_ids, test_ids, pred_ids = [], [], [], []
split_root = Path(DATA_ROOT) / 'split'

def load_ids(name, lst):
    with open(split_root / f'{name}.txt', 'r') as f:
        for line in f:
            lst.append(line.strip())

load_ids('train', train_ids)
load_ids('val',   val_ids)
load_ids('test',  test_ids)
load_ids('pred',  pred_ids)

print(f"Train : {len(train_ids)} | Val : {len(val_ids)} | Test : {len(test_ids)} | Pred : {len(pred_ids)}")

Train : 59 | Val : 10 | Test : 10 | Pred : 19


# Physics Informed Feature Engineering

We compute two additional channels from the raw bands:
- **NDWI** = (Green − NIR) / (Green + NIR + epsilon)  →  highlights water (positive values = water)
- **SAR HH/HV ratio** = HH / (HV + epsilon)  →  surface roughness proxy; open water has low HV backscatter

In [5]:
def compute_derived_channels(raw: np.ndarray) -> np.ndarray:
    """
    raw : float32 array of shape [6, H, W] with bands in order
          [SAR_HH, SAR_HV, Green, Red, NIR, SWIR]
    Returns float32 array of shape [8, H, W] with two extra channels:
          [..., NDWI, SAR_HH_HV_ratio]
    """
    eps = 1e-6
    green = raw[BAND_GREEN]
    nir   = raw[BAND_NIR]
    hh    = raw[BAND_SAR_HH]
    hv    = raw[BAND_SAR_HV]

    # NDWI — clamped to [-1, 1]
    ndwi = (green - nir) / (green + nir + eps)
    ndwi = np.clip(ndwi, -1.0, 1.0)

    # SAR HH/HV ratio (log-space to compress dynamic range)
    sar_ratio = np.log(np.abs(hh) + eps) - np.log(np.abs(hv) + eps)

    return np.concatenate([raw, ndwi[None], sar_ratio[None]], axis=0)  # [8, H, W]

# Dataset

In [6]:
class FloodDataset(Dataset):
    """
    Loads 6-channel GeoTIFF images, computes physics-informed features to
    produce 8-channel tensors, and returns 3-class segmentation masks.

    Label convention (assumed):
        0 = Background / No-flood
        1 = Flood
        2 = Permanent Water Body
       -1 = Ignore (unclassified)
    """
    def __init__(self, split_name, data_root, transform=None, is_predict=False):
        self.data_root  = Path(data_root)
        self.transform  = transform
        self.is_predict = is_predict

        with open(self.data_root / f'split/{split_name}.txt', 'r') as f:
            self.file_ids = [line.strip() for line in f]

        if is_predict:
            self.image_dir = self.data_root / 'prediction/image'
            self.label_dir = None
        else:
            self.image_dir = self.data_root / 'image'
            self.label_dir = self.data_root / 'label'

    def __len__(self):
        return len(self.file_ids)

    def __getitem__(self, idx):
        fid = self.file_ids[idx]

        # ── Load raw 6-channel image ────────────────────────────────────────
        img_path = list(self.image_dir.glob(f'*{fid}_image.tif'))[0]
        with rasterio.open(img_path) as src:
            raw = src.read().astype(np.float32)      # [6, H, W]

        # ── Compute 8-channel physics-informed tensor ───────────────────────
        image8 = compute_derived_channels(raw)       # [8, H, W]

        # ── Z-Score Normalisation ───────────────────────────────────────────
        image8 = (image8 - MEANS_8[:, None, None]) / STDS_8[:, None, None]
        image_tensor = torch.from_numpy(image8).float()

        if self.is_predict:
            if self.transform:
                image_tensor = self.transform(image_tensor)
            return image_tensor, str(img_path)

        # ── Load label mask ─────────────────────────────────────────────────
        lbl_path = list(self.label_dir.glob(f'*{fid}_label.tif'))[0]
        with rasterio.open(lbl_path) as src:
            mask = src.read(1).astype(np.int64)      # [H, W]

        mask_tensor = torch.from_numpy(mask)

        # ── Augmentation (torchvision v2 API for aligned spatial transforms) ─
        if self.transform:
            mask_tv = tv_tensors.Mask(mask_tensor)
            image_tensor, mask_tensor = self.transform(image_tensor, mask_tv)

        return image_tensor, mask_tensor.long()

# Datamodule

In [7]:
class FloodDataModule(pl.LightningDataModule):
    def __init__(self, data_root, batch_size=4):
        super().__init__()
        self.data_root  = data_root
        self.batch_size = batch_size

        # ─── Heavy Augmentation Pipeline (Safe for 8-Channel Tensors) ─────────
        self.train_transform = v2.Compose([
            # 1. D4 Symmetry (Covers all basic orientations of satellite data)
            v2.RandomHorizontalFlip(p=0.5),
            v2.RandomVerticalFlip(p=0.5),
            v2.RandomApply([
                v2.RandomChoice([
                    v2.RandomRotation(degrees=(90, 90)),
                    v2.RandomRotation(degrees=(180, 180)),
                    v2.RandomRotation(degrees=(270, 270))
                ])
            ], p=0.6), # 60% chance to apply one of the 3 strict rotations

            # 2. Geometric Distortions (Combats spatial memorization)
            v2.RandomApply([
                # Shifts, zooms, and shears the image slightly
                v2.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.85, 1.15), shear=10)
            ], p=0.5),
            
            v2.RandomApply([
                # Simulates topographical/perspective warps
                v2.ElasticTransform(alpha=50.0, sigma=5.0)
            ], p=0.3),

            # 3. Pixel-Level Corruption
            v2.RandomApply([v2.GaussianNoise(mean=0.0, sigma=0.05)], p=0.3),
            
            # Cutout / Random Erasing: Drops random blocks of data.
            v2.RandomErasing(p=0.4, scale=(0.02, 0.15), ratio=(0.3, 3.3), value=0)
        ])
        
        self.val_transform = None

    def setup(self, stage=None):
        if stage in ('fit', None):
            self.train_ds = FloodDataset('train', self.data_root, self.train_transform)
            self.val_ds   = FloodDataset('val',   self.data_root, self.val_transform)
        if stage in ('predict', None):
            self.pred_ds  = FloodDataset('pred',  self.data_root, self.val_transform, is_predict=True)

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True,
                          num_workers=2, pin_memory=True, persistent_workers=True)

    def val_dataloader(self):
        return DataLoader(self.val_ds, batch_size=self.batch_size, shuffle=False,
                          num_workers=2, pin_memory=True, persistent_workers=True)

    def predict_dataloader(self):
        return DataLoader(self.pred_ds, batch_size=1, shuffle=False, num_workers=2)

# Model

In [8]:
class ResNetEncoder(nn.Module):
    def __init__(self, in_channels=8, pretrained=True):
        super().__init__()
        self.body = timm.create_model('resnet18', pretrained=pretrained, features_only=True)
        
        # Patch the first convolution for 8 channels
        old_conv = self.body.conv1
        new_conv = nn.Conv2d(in_channels, old_conv.out_channels, 
                             kernel_size=old_conv.kernel_size, stride=old_conv.stride,
                             padding=old_conv.padding, bias=False)
        with torch.no_grad():
            if pretrained:
                # Copy ImageNet weights for Green/Red/NIR
                new_conv.weight[:, :3, :, :] = old_conv.weight.data[:, :3, :, :]
            # Kaiming init for the rest
            nn.init.kaiming_normal_(new_conv.weight[:, 3:, :, :], mode='fan_out', nonlinearity='relu')
            
        self.body.conv1 = new_conv
        self.out_channels = self.body.feature_info.channels() 

    def forward(self, x):
        features = self.body(x)
        return features[1], features[2], features[3], features[4]

class LinkDecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout_p=0.35):
        super().__init__()
        self.upsample  = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.proj_up   = nn.Conv2d(in_channels,   out_channels, 1, bias=False)
        self.proj_skip = nn.Conv2d(skip_channels, out_channels, 1, bias=False)
        self.refine    = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            
            # ─── ANTI-MEMORIZATION WEAPON ───
            # Drops 35% of the spatial feature maps randomly. 
            nn.Dropout2d(p=dropout_p), 
            
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x, skip):
        x = self.upsample(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)
        return self.refine(self.proj_up(x) + self.proj_skip(skip))

class ADLinkNet(nn.Module):
    def __init__(self, in_channels=8, num_classes=3, pretrained=True):
        super().__init__()
        self.encoder = ResNetEncoder(in_channels=in_channels, pretrained=pretrained)
        c = self.encoder.out_channels 

        self.dec4 = LinkDecoderBlock(c[4], c[3], 256)   
        self.dec3 = LinkDecoderBlock(256,  c[2], 128)   
        self.dec2 = LinkDecoderBlock(128,  c[1], 64)    

        self.final_up = nn.Sequential(
            nn.Conv2d(64, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=0.2), # Lighter dropout right before the final prediction
        )
        self.head = nn.Conv2d(64, num_classes, 1)

    def forward(self, x):
        input_size = x.shape[-2:] 
        
        f1, f2, f3, f4 = self.encoder(x)
        out = self.dec4(f4, f3)
        out = self.dec3(out,  f2)
        out = self.dec2(out,  f1)
        
        out = F.interpolate(out, size=input_size, mode='bilinear', align_corners=False)
        out = self.final_up(out)
        return self.head(out)

# Loss Function

In [9]:
# ─── Loss Components ──────────────────────────────────────────────────────────

class WeightedFocalLoss(nn.Module):
    """
    Multi-class Focal Loss with per-class weights.
    Flood (class 1) gets highest weight to force the model to detect it.
    """
    def __init__(self, gamma=2.0, class_weights=None, ignore_index=-1):
        super().__init__()
        self.gamma        = gamma
        self.class_weights = class_weights   # Tensor of length num_classes
        self.ignore_index  = ignore_index

    def forward(self, logits, targets):
        ce = F.cross_entropy(
            logits, targets,
            weight=self.class_weights.to(logits.device) if self.class_weights is not None else None,
            ignore_index=self.ignore_index,
            reduction='none'
        )
        pt     = torch.exp(-ce)
        focal  = (1 - pt) ** self.gamma * ce
        valid  = targets != self.ignore_index
        return focal[valid].mean()


class MultiClassDiceLoss(nn.Module):
    """
    Soft Dice loss averaged over all classes (excluding ignore).
    Directly optimises the mIoU metric we are graded on.
    """
    def __init__(self, num_classes=3, ignore_index=-1, smooth=1.0):
        super().__init__()
        self.num_classes  = num_classes
        self.ignore_index = ignore_index
        self.smooth       = smooth

    def forward(self, logits, targets):
        valid = (targets != self.ignore_index)
        tgt   = targets.clone()
        tgt[~valid] = 0

        probs = F.softmax(logits, dim=1)   # [B, C, H, W]
        tgt_oh = F.one_hot(tgt, self.num_classes).permute(0, 3, 1, 2).float()  # [B, C, H, W]
        v_mask = valid.unsqueeze(1).float()

        dice_per_class = []
        for c in range(self.num_classes):
            p = probs[:, c] * valid.float()
            g = tgt_oh[:, c] * valid.float()
            inter = (p * g).sum(dim=(1, 2))
            denom = (p + g).sum(dim=(1, 2))
            dice  = (2 * inter + self.smooth) / (denom + self.smooth)
            dice_per_class.append(1 - dice.mean())

        return torch.stack(dice_per_class).mean()


class BoundaryLoss(nn.Module):
    """
    Penalises prediction errors specifically at class boundaries.
    Uses a distance-transform map computed from the ground-truth mask:
    pixels close to a boundary have higher weight.

    Reference: Kervadec et al., "Boundary loss for highly unbalanced segmentation", MIDL 2019.
    """
    def __init__(self, ignore_index=-1):
        super().__init__()
        self.ignore_index = ignore_index

    @staticmethod
    def _dist_map_batch(masks: torch.Tensor, num_classes: int) -> torch.Tensor:
        """
        Compute signed distance maps for each class in the batch.
        Returns shape [B, C, H, W] on CPU, values in [−1, 1].
        """
        B, H, W = masks.shape
        dist = torch.zeros(B, num_classes, H, W)
        masks_np = masks.cpu().numpy()
        for b in range(B):
            for c in range(num_classes):
                fg = (masks_np[b] == c).astype(np.float32)
                if fg.sum() == 0:
                    continue
                d_in  = distance_transform_edt(fg)
                d_out = distance_transform_edt(1 - fg)
                sdf   = d_in - d_out
                mx    = np.abs(sdf).max() + 1e-6
                dist[b, c] = torch.from_numpy(sdf / mx)
        return dist

    def forward(self, logits, targets):
        valid = (targets != self.ignore_index)
        tgt   = targets.clone()
        tgt[~valid] = 0

        num_classes = logits.shape[1]
        probs = F.softmax(logits, dim=1)                           # [B, C, H, W]
        dist  = self._dist_map_batch(tgt, num_classes).to(logits.device)  # [B, C, H, W]

        # Boundary loss = mean of (prob × signed_dist_field)
        # Correct predictions (prob → 1 inside, → 0 outside) minimise this
        bl = (probs * dist).sum(dim=1)   # [B, H, W]
        return bl[valid].mean()

# (Keep WeightedFocalLoss, MultiClassDiceLoss, and BoundaryLoss as they were)

class TriLoss(nn.Module):
    """
    Combined loss:  α·Focal + β·Dice + γ·Boundary
    """
    def __init__(self, num_classes=3, ignore_index=-1,
                 alpha=0.4, beta=0.4, gamma_bnd=0.2):
        super().__init__()
        self.alpha     = alpha
        self.beta      = beta
        self.gamma_bnd = gamma_bnd

        # class_w = torch.tensor([1.0, 3.0, 2.0])
        class_w = torch.tensor([0.5, 10.0, 3.0]) 
        
        self.focal    = WeightedFocalLoss(gamma=2.0, class_weights=class_w, ignore_index=ignore_index)
        self.dice     = MultiClassDiceLoss(num_classes=num_classes, ignore_index=ignore_index)
        self.boundary = BoundaryLoss(ignore_index=ignore_index)

    def forward(self, logits, targets):
        l_focal = self.focal(logits, targets)
        l_dice  = self.dice(logits, targets)
        l_bnd   = self.boundary(logits, targets)
        return self.alpha * l_focal + self.beta * l_dice + self.gamma_bnd * l_bnd

# Tracking Metric

In [10]:
class FloodSegmentationModelV3(pl.LightningModule):
    """
    AD-LinkNet + ResNet18, 4-channel input, 3-class output.
    Tracks per-class IoU and mIoU for validation.
    """
    def __init__(self,
                 in_channels=8,       # Now defaults to 4
                 num_classes=3,
                 learning_rate=3e-4,
                 pretrained=True):
        super().__init__()
        self.save_hyperparameters()
        self.learning_rate = learning_rate
        self.num_classes   = num_classes

        # Initialize the new ResNet-based model
        self.model     = ADLinkNet(in_channels=in_channels, num_classes=num_classes, pretrained=pretrained)
        self.criterion = TriLoss(num_classes=num_classes)

        # Accumulate TP/FP/FN across batches for epoch-level IoU
        self._reset_iou_stats('val')
        self._reset_iou_stats('train')

    # ── Helpers ────────────────────────────────────────────────────────────────
    def _reset_iou_stats(self, split):
        C = self.num_classes
        setattr(self, f'{split}_tp', torch.zeros(C))
        setattr(self, f'{split}_fp', torch.zeros(C))
        setattr(self, f'{split}_fn', torch.zeros(C))

    def _compute_iou_stats(self, logits, masks):
        preds = torch.argmax(logits, dim=1)
        valid = masks != -1
        tp = torch.zeros(self.num_classes, device=self.device)
        fp = torch.zeros(self.num_classes, device=self.device)
        fn = torch.zeros(self.num_classes, device=self.device)
        for c in range(self.num_classes):
            pred_c = (preds == c) & valid
            true_c = (masks == c) & valid
            tp[c]  = (pred_c & true_c).sum()
            fp[c]  = (pred_c & ~true_c).sum()
            fn[c]  = (~pred_c & true_c).sum()
        return tp, fp, fn

    def _log_epoch_iou(self, split):
        tp = getattr(self, f'{split}_tp')
        fp = getattr(self, f'{split}_fp')
        fn = getattr(self, f'{split}_fn')
        iou  = tp / (tp + fp + fn + 1e-6)
        miou = iou.mean()
        self.log(f'{split}_mIoU',       miou,    prog_bar=True)
        self.log(f'{split}_IoU_BG',     iou[0],  prog_bar=False)
        self.log(f'{split}_IoU_Flood',  iou[1],  prog_bar=True)
        self.log(f'{split}_IoU_Water',  iou[2],  prog_bar=True)
        self._reset_iou_stats(split)

    # ── Forward ────────────────────────────────────────────────────────────────
    def forward(self, x):
        return self.model(x)

    # ── Training ───────────────────────────────────────────────────────────────
    def training_step(self, batch, batch_idx):
        images, masks = batch
        logits = self(images)
        loss   = self.criterion(logits, masks)
        tp, fp, fn = self._compute_iou_stats(logits, masks)
        self.train_tp += tp.cpu()
        self.train_fp += fp.cpu()
        self.train_fn += fn.cpu()
        self.log('train_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def on_train_epoch_end(self):
        self._log_epoch_iou('train')

    # ── Validation ─────────────────────────────────────────────────────────────
    def validation_step(self, batch, batch_idx):
        images, masks = batch
        logits = self(images)
        loss   = self.criterion(logits, masks)
        tp, fp, fn = self._compute_iou_stats(logits, masks)
        self.val_tp += tp.cpu()
        self.val_fp += fp.cpu()
        self.val_fn += fn.cpu()
        self.log('val_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def on_validation_epoch_end(self):
        self._log_epoch_iou('val')

    # ── Prediction ─────────────────────────────────────────────────────────────
    def predict_step(self, batch, batch_idx):
        images, paths = batch
        logits = self(images)
        return torch.argmax(logits, dim=1), paths

    # ── Optimiser (The New Frozen ResNet Code) ─────────────────────────────────
    def configure_optimizers(self):
        # 1. Freeze the early layers of ResNet to force it to use general features
        for param in self.model.encoder.body.conv1.parameters(): param.requires_grad = False
        for param in self.model.encoder.body.bn1.parameters(): param.requires_grad = False
        for param in self.model.encoder.body.layer1.parameters(): param.requires_grad = False
        
        # 2. Gather only the unfrozen backbone params
        trainable_backbone = [p for p in self.model.encoder.parameters() if p.requires_grad]
        
        decoder_params  = (
            list(self.model.dec4.parameters()) +
            list(self.model.dec3.parameters()) +
            list(self.model.dec2.parameters()) +
            list(self.model.final_up.parameters()) +
            list(self.model.head.parameters())
        )
        
        # 3. Massive increase to weight_decay (1e-4 -> 1e-2)
        optimizer = torch.optim.AdamW([
            {'params': trainable_backbone, 'lr': self.learning_rate * 0.1},
            {'params': decoder_params,  'lr': self.learning_rate},
        ], weight_decay=1e-2) 

        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=10, T_mult=1, eta_min=1e-6
        )
        return {
            'optimizer': optimizer,
            'lr_scheduler': {'scheduler': scheduler, 'interval': 'epoch'}
        }

# Training

In [11]:
# pl.seed_everything(42)

# datamodule = FloodDataModule(data_root=DATA_ROOT, batch_size=BATCH_SIZE)
# model      = FloodSegmentationModelV3(
#     in_channels=IN_CHANNELS,
#     num_classes=NUM_CLASSES,
#     learning_rate=LR,
#     pretrained=True,
# )

# checkpoint_callback = ModelCheckpoint(
#     dirpath=MODEL_SAVE_DIR,
#     filename='best-adlinknet-resnet18-{epoch:02d}-{val_mIoU:.4f}',
#     save_top_k=3,
#     monitor='val_mIoU',
#     mode='max',
# )
# lr_monitor = LearningRateMonitor(logging_interval='epoch')

# trainer = pl.Trainer(
#     max_epochs=EPOCHS,
#     accelerator='auto',
#     devices=1,
#     precision='16-mixed',
#     accumulate_grad_batches=2,    # FIX: Effective batch size = 8 x 2 = 16
#     gradient_clip_val=1.0,
#     log_every_n_steps=2,          # FIX: Lowered so it actually logs with the small step count
#     callbacks=[checkpoint_callback, lr_monitor],
# )

# print("Starting Training — AD-LinkNet + ResNet18 + 8ch + Tri-Loss")
# trainer.fit(model, datamodule=datamodule)

# TTA

In [12]:
# ─── 4. Run TTA Inference (Corrected) ─────────────────────────────────────────
def predict_with_tta(model, dataloader, device):
    model.eval()
    all_preds, all_paths = [], []

    with torch.no_grad():
        for images, paths in dataloader:
            images = images.to(device)

            aug_probs = [
                torch.softmax(model(images), dim=1),
                torch.softmax(model(torch.flip(images, [3])), dim=1).flip([3]),
                torch.softmax(model(torch.flip(images, [2])), dim=1).flip([2]),
                torch.softmax(model(torch.flip(images, [2, 3])), dim=1).flip([2, 3]),
            ]
            avg_probs = torch.stack(aug_probs).mean(0)
            preds     = torch.argmax(avg_probs, dim=1)  # [B, H, W]

            # SWIR post-processing REMOVED to prevent cross-class contamination
            all_preds.extend(preds.cpu().unbind(0))
            all_paths.extend(paths)

    return all_preds, all_paths

# # Load best checkpoint
# best_ckpt  = checkpoint_callback.best_model_path
# loaded_mdl = FloodSegmentationModelV3.load_from_checkpoint(best_ckpt)
# loaded_mdl.eval()
# device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# loaded_mdl = loaded_mdl.to(device)

# datamodule.setup('predict')
# pred_loader = datamodule.predict_dataloader()

# tta_preds, tta_paths = predict_with_tta(loaded_mdl, pred_loader, device)
# print(f"TTA inference complete — {len(tta_preds)} samples")

# Submission

In [13]:
def mask_to_rle(mask: np.ndarray) -> str:
    """
    Converts a binary mask to RLE (column-major, 1-indexed).
    If the mask is empty, returns '0 0' as per competition rules.
    """
    pixels = mask.flatten(order='F')
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    
    rle_string = ' '.join(str(x) for x in runs)
    
    # Return '0 0' if there are no flood pixels detected
    return rle_string if rle_string else '0 0'

# submission_rows = []
# for pred_tensor, file_path in zip(tta_preds, tta_paths):
#     preds_np = pred_tensor.numpy()                         # [H, W] with values 0,1,2
#     img_id   = Path(file_path).name.replace('_image.tif', '')
#     # RLE on Flood class (1)
#     flood_mask = (preds_np == 1).astype(np.uint8)
#     rle_string = mask_to_rle(flood_mask)
#     submission_rows.append({'id': img_id, 'rle_mask': rle_string})

# df_sub = pd.DataFrame(submission_rows)
# df_sub = df_sub.replace('', 0).fillna(0)
# sub_path = f'{OUT_DIR}/submission_adlinknet_tta.csv'
# df_sub.to_csv(sub_path, index=False)
# print(f'Submission saved: {sub_path}')
# df_sub.head()

# Kaggle Hub Upload

In [14]:
# USERNAME   = "devamwppu2000828"
# MODEL_SLUG = "aise-flood-adlinknet-resnet18"
# FRAMEWORK  = "pytorch"
# VARIATION  = "v1-finetuned"
# handle     = f"{USERNAME}/{MODEL_SLUG}/{FRAMEWORK}/{VARIATION}"

# print(f"Uploading to Kaggle Hub: {handle}")
# try:
#     kagglehub.model_upload(
#         handle=handle,
#         local_model_dir=MODEL_SAVE_DIR,
#         version_notes="AD-LinkNet + Xception + 8ch physics-informed + tri-loss"
#     )
#     print("Successfully uploaded!")
# except Exception as e:
#     print(f"Upload failed: {e}")

# Inference

In [15]:
import os
import re
import torch
import kagglehub
import pandas as pd
from pathlib import Path

# ─── 1. Download Model from Kaggle Hub ───────────────────────────────────────
print("Downloading model from Kaggle Hub...")
# Note: kagglehub.model_download returns the path where the files are mounted
model_dir = kagglehub.model_download("devamwppu2000828/aise-flood-adlinknet-resnet18/pytorch/v1")
print(f"Model downloaded to: {model_dir}")

# ─── 2. Auto-Select the Best Checkpoint from the DOWNLOADED dir ──────────────
# CHANGE: We now look inside 'model_dir' instead of 'MODEL_SAVE_DIR'
ckpt_files = [f for f in os.listdir(model_dir) if f.endswith('.ckpt') and 'resnet18' in f]

if not ckpt_files:
    # If no 'resnet18' string in filename, fallback to all .ckpt files in that dir
    ckpt_files = [f for f in os.listdir(model_dir) if f.endswith('.ckpt')]

if not ckpt_files:
    raise FileNotFoundError(f"No checkpoints found in {model_dir}")

def extract_miou(filename):
    match = re.search(r'val_mIoU[=]?([0-9]+\.[0-9]+)', filename)
    return float(match.group(1)) if match else 0.0

# Sort and pick the best one from the downloaded files
ckpt_files.sort(key=extract_miou, reverse=True)
best_ckpt_path = os.path.join(model_dir, ckpt_files[0])

print(f"Correctly identified best checkpoint: {best_ckpt_path}")

# ─── 3. Load the Model ───────────────────────────────────────────────────────
loaded_mdl = FloodSegmentationModelV3.load_from_checkpoint(best_ckpt_path)
loaded_mdl.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
loaded_mdl = loaded_mdl.to(device)

# ─── 4. Run TTA Inference ────────────────────────────────────────────────────
datamodule = FloodDataModule(data_root=DATA_ROOT, batch_size=BATCH_SIZE)
datamodule.setup('predict')
pred_loader = datamodule.predict_dataloader()

print("Starting TTA Inference...")
tta_preds, tta_paths = predict_with_tta(loaded_mdl, pred_loader, device)
print(f"TTA inference complete — {len(tta_preds)} samples processed.")

# ─── 5. Generate Submission RLE ──────────────────────────────────────────────
submission_rows = []
for pred_tensor, file_path in zip(tta_preds, tta_paths):
    preds_np = pred_tensor.numpy()
    img_id   = Path(file_path).name.replace('_image.tif', '')
    
    flood_mask = (preds_np == 1).astype(np.uint8)
    rle_string = mask_to_rle(flood_mask)
    
    submission_rows.append({'id': img_id, 'rle_mask': rle_string})

# ─── 6. Save Submission ──────────────────────────────────────────────────────
df_sub = pd.DataFrame(submission_rows)
df_sub['rle_mask'] = df_sub['rle_mask'].replace('', '0 0').fillna('0 0')

sub_path = '/kaggle/working/submission.csv'
df_sub.to_csv(sub_path, index=False)

print(f"Submission saved as {sub_path}")
df_sub.head()

Model downloaded to: /kaggle/input/models/devamwppu2000828/aise-flood-adlinknet-resnet18/pytorch/v1/1
Correctly identified best checkpoint: /kaggle/input/models/devamwppu2000828/aise-flood-adlinknet-resnet18/pytorch/v1/1/best-adlinknet-resnet18-epoch10-val_mIoU0.4004.ckpt


model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

Starting TTA Inference...
TTA inference complete — 19 samples processed.
Submission saved as /kaggle/working/submission.csv


,id,rle_mask
0,20240529_EO4_RES2_fl_pid_080,99 7 415 6 460 14 495 18 613 4 973 12 1008 11 ...
1,20240529_EO4_RES2_fl_pid_081,45 3 113 8 268 7 338 32 383 7 421 32 469 16 50...
2,20240529_EO4_RES2_fl_pid_082,1 6 139 34 210 2 376 11 467 17 493 24 656 26 8...
3,20240529_EO4_RES2_fl_pid_083,1 33 47 38 170 8 394 14 513 33 559 37 907 12 1...
4,20240529_EO4_RES2_fl_pid_084,19 6 86 5 115 11 135 5 218 41 261 185 493 8 50...
